# Utmaning: Analysera text om data science

I detta exempel gör vi en enkel övning som täcker alla steg i en traditionell data science-process. Du behöver inte skriva någon kod, du kan bara klicka på cellerna nedan för att köra dem och observera resultatet. Som en utmaning uppmanas du att testa denna kod med olika data.

## Mål

I denna lektion har vi diskuterat olika begrepp relaterade till Data Science. Låt oss försöka upptäcka fler relaterade begrepp genom att göra lite **textmining**. Vi börjar med en text om Data Science, extraherar nyckelord från den och försöker sedan visualisera resultatet.

Som text kommer jag att använda sidan om Data Science från Wikipedia:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Steg 1: Hämta data

Det första steget i varje data science-process är att hämta data. Vi kommer att använda biblioteket `requests` för att göra det:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Steg 2: Omvandla data

Nästa steg är att konvertera datan till en form som är lämplig för bearbetning. I vårt fall har vi laddat ner HTML-källkoden från sidan, och vi behöver omvandla den till ren text.

Det finns många sätt att göra detta på. Vi kommer att använda [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), ett populärt Python-bibliotek för att analysera HTML. BeautifulSoup gör det möjligt för oss att rikta in oss på specifika HTML-element, så att vi kan fokusera på huvudinnehållet i artikeln från Wikipedia och minska ner på vissa navigationsmenyer, sidofält, sidfötter och annat irrelevanta innehåll (även om en del standardtext kan finnas kvar).


Först behöver vi installera BeautifulSoup-biblioteket för HTML-parsning:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Steg 3: Få insikter

Det viktigaste steget är att omvandla vår data till någon form från vilken vi kan dra insikter. I vårt fall vill vi extrahera nyckelord från texten och se vilka nyckelord som är mer meningsfulla.

Vi kommer att använda ett Python-bibliotek som heter [RAKE](https://github.com/aneesha/RAKE) för nyckelordsextraktion. Först låt oss installera detta bibliotek om det inte finns: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Huvudfunktionen är tillgänglig från `Rake`-objektet, vilket vi kan anpassa med några parametrar. I vårt fall kommer vi att sätta minimilängden för ett nyckelord till 5 tecken, minimifrekvensen för ett nyckelord i dokumentet till 3, och maximalt antal ord i ett nyckelord till 2. Känn dig fri att experimentera med andra värden och observera resultatet.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Vi erhöll en lista med termer tillsammans med tillhörande grad av viktighet. Som du kan se, är de mest relevanta disciplinerna, såsom maskininlärning och big data, närvarande i listan på toppositioner.

## Steg 4: Visualisera resultatet

Människor kan bäst tolka data i visuell form. Därför är det ofta meningsfullt att visualisera data för att dra några insikter. Vi kan använda `matplotlib`-biblioteket i Python för att plotta enkel fördelning av nyckelorden med deras relevans:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Det finns dock ett ännu bättre sätt att visualisera ordfrekvenser – med hjälp av **Word Cloud**. Vi kommer behöva installera ett annat bibliotek för att rita ordmolnet från vår nyckelordslista.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud`-objektet ansvarar för att ta emot antingen originaltext eller en förberäknad lista över ord med deras frekvenser, och returnerar en bild som sedan kan visas med `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Vi kan också skicka in den ursprungliga texten till `WordCloud` - låt oss se om vi kan få ett liknande resultat:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Du kan se att ordmolnet nu ser mer imponerande ut, men det innehåller också mycket brus (t.ex. orelaterade ord som `Retrieved on`). Dessutom får vi färre nyckelord som består av två ord, såsom *data scientist* eller *computer science*. Detta beror på att RAKE-algoritmen gör ett mycket bättre jobb med att välja ut bra nyckelord från texten. Detta exempel illustrerar vikten av datarensning och förbehandling, eftersom en klar bild i slutändan gör att vi kan fatta bättre beslut.

I denna övning har vi gått igenom en enkel process för att extrahera viss mening från Wikipediatext, i form av nyckelord och ordmoln. Detta exempel är ganska enkelt, men det visar väl alla typiska steg som en data scientist tar när hen jobbar med data, från dataförvärv fram till visualisering.

I vår kurs kommer vi att diskutera alla dessa steg i detalj.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
